In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/diabetes_raw.csv")
df.shape

(253680, 22)

## Handling Duplicate Records

During inspection, 24,206 duplicate rows (~9.5% of the dataset) were identified.
Investigation showed these duplicates form small clusters (median cluster size = 2),
scattered across non-adjacent row indices, and the largest clusters consist of
common "low-risk" survey response patterns (e.g. no reported health conditions,
active lifestyle, normal BMI range). Given the dataset's mostly binary/low-cardinality
feature space and large sample size, this pattern is consistent with different
survey respondents coincidentally sharing identical answers, rather than a data
collection or export error.

**Decision:** Duplicate rows will be retained, since removing them would risk
disproportionately discarding genuine respondents with common health profiles,
which could bias the class distribution further.

In [2]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows identified: {duplicate_count} ({duplicate_count/len(df)*100:.2f}% of dataset)")
print("Decision: retained (see markdown above for justification)")

Duplicate rows identified: 24206 (9.54% of dataset)
Decision: retained (see markdown above for justification)


## Handling BMI Outliers

Inspection showed BMI values ranging from 12 to 98. Both tails were examined:
the lowest values (12-13) and highest values (95-98) formed smooth, gradually
declining distributions rather than isolated freak values, and were scattered
across different row indices rather than clustered together. While medically
extreme, values in this range are physiologically possible (severe underweight
at the low end, severe obesity at the high end) and show no signs of data-entry
errors (e.g. no zero, negative, or implausibly large values like 300+).

**Decision:** BMI outliers will be retained rather than removed or capped, since
they likely represent genuine extreme cases in a large population survey.
They will be clearly visualised (e.g. boxplot) in the EDA stage so their
influence on the data is transparent rather than hidden.

In [3]:
print(f"BMI range: {df['BMI'].min()} to {df['BMI'].max()}")
print("Decision: outliers retained (see markdown above for justification)")

BMI range: 12 to 98
Decision: outliers retained (see markdown above for justification)


## Converting Ordinal Variables to Categorical Type

Age, Education, Income, and GenHlth are stored as integers, but represent
ordinal bracket/category codes rather than continuous numeric values (e.g.
Age = 9 refers to an age bracket, not literally 9 years old). To make this
explicit in the data and prevent unintended numeric operations (e.g. taking
a mean of Age brackets as if it were a real age), these columns are converted
to pandas' ordered categorical dtype.

In [4]:
ordinal_cols = ['Age', 'Education', 'Income', 'GenHlth']

for col in ordinal_cols:
    df[col] = pd.Categorical(df[col], ordered=True)

df[ordinal_cols].dtypes

Age          category
Education    category
Income       category
GenHlth      category
dtype: object

In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/diabetes_cleaned.csv", index=False)
print("Cleaned dataset saved to data/processed/diabetes_cleaned.csv")

Cleaned dataset saved to data/processed/diabetes_cleaned.csv


## Summary of Task 3 — Cleaning & Transformation Decisions

| Issue | Finding | Decision | Rationale |
|---|---|---|---|
| Duplicate rows | 24,206 rows (9.54%) | Retained | Small, scattered clusters consistent with coincidental overlap in a low-cardinality feature space, not data errors |
| BMI outliers | Range 12-98 | Retained | Smooth distribution tails, no signs of data-entry errors, medically extreme but plausible |
| Ordinal columns (Age, Education, Income, GenHlth) | Stored as int64 | Converted to ordered `category` dtype | Prevents misuse as continuous numeric values |
| Missing values | None found | No action needed | Dataset confirmed fully populated during Task 2 inspection |

**Output:** Cleaned dataset saved to `data/processed/diabetes_cleaned.csv`,
ready for exploratory data analysis (Task 4).